# Diagnostic — Inspect Failed Trials

29 trials failed during preprocessing. This notebook inspects them to determine if they are **corrupted** (truncated/malformed data) or merely **unusual** (odd number of columns, unexpected delimiter, etc).

Based on the error messages, they look truncated mid-number in scientific notation — classic signature of an incomplete file write during dataset distribution.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from config import METADATA_PATH
from src.preprocessing import preprocess_trial

# Re-identify the 29 failed trials by re-running preprocessing on all rows
# and catching exceptions (without actually saving anything this time)
metadata = pd.read_csv(METADATA_PATH)

failed = []
for idx, row in metadata.iterrows():
    try:
        preprocess_trial(row["raw_file_path"], int(row["gesture_label"]))
    except Exception as exc:
        failed.append({
            "subject": row["subject"],
            "session": row["session"],
            "position": row["position"],
            "repetition": row["repetition"],
            "gesture": row["gesture"],
            "path": row["raw_file_path"],
            "error": str(exc)[:80],
        })

failed_df = pd.DataFrame(failed)
print(f"Total failed: {len(failed_df)}")
failed_df

Total failed: 29


,subject,session,position,repetition,gesture,path,error
0,h0,8,1,2,open_hand,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '6.00000000...
1,h0,8,5,2,eversion,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '3.60000000...
2,h1,0,1,2,pinch_middlefinger,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '-'
3,h1,0,7,2,two,/Users/erdiantiwigaputriandini/Documents/Kulia...,No columns to parse from file
4,h1,1,2,1,pinch_forefinger,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '-1.1000000...
5,h1,4,0,1,fist,/Users/erdiantiwigaputriandini/Documents/Kulia...,No columns to parse from file
6,h10,2,3,1,varus,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '-4.9000000...
7,h13,1,2,0,fist,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '2.50000000...
8,h2,4,10,2,eversion,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '-6.0000000...
9,h2,5,7,0,pinch_forefinger,/Users/erdiantiwigaputriandini/Documents/Kulia...,could not convert string to float: '-2.0000000...


## Inspect file sizes and last lines

A well-formed trial CSV is 6-9 seconds × 200 Hz ≈ 1200-1800 rows, 8 floats per row → ~30-60 KB typical. Corrupted files will be smaller or have weird endings.

In [2]:
import os

# Pick a reference healthy trial (first one) for comparison
reference_path = metadata.iloc[0]["raw_file_path"]
ref_size = os.path.getsize(reference_path)
print(f"Reference healthy trial: {Path(reference_path).name}")
print(f"  Size: {ref_size:,} bytes")
with open(reference_path) as f:
    ref_lines = f.readlines()
print(f"  Lines: {len(ref_lines)}")
print(f"  Last 2 lines:\n    {ref_lines[-2].strip()}\n    {ref_lines[-1].strip()}")
print()

print("=" * 70)
print("FAILED TRIALS — size and last line inspection")
print("=" * 70)

for _, row in failed_df.iterrows():
    path = row["path"]
    size = os.path.getsize(path)
    try:
        with open(path) as f:
            lines = f.readlines()
        n_lines = len(lines)
        last_line = lines[-1].strip() if lines else "<empty>"
    except Exception as e:
        n_lines = "?"
        last_line = f"<read error: {e}>"

    print(f"\n{row['subject']}/s{row['session']}/p{row['position']}/r{row['repetition']}/{row['gesture']}")
    print(f"  Size:      {size:,} bytes ({100 * size / ref_size:.1f}% of reference)")
    print(f"  Lines:     {n_lines}")
    print(f"  Last line: {last_line[:100]}")

Reference healthy trial: emg_p0_r0_eversion.csv
  Size: 334,050 bytes
  Lines: 1628
  Last 2 lines:
    2.600000000000000000e+01,-2.000000000000000000e+00,-1.000000000000000000e+00,6.000000000000000000e+00,1.500000000000000000e+01,3.000000000000000000e+00,7.000000000000000000e+00,1.060000000000000000e+02
    -2.600000000000000000e+01,0.000000000000000000e+00,-1.000000000000000000e+00,-2.000000000000000000e+00,-2.000000000000000000e+01,7.000000000000000000e+00,6.000000000000000000e+01,3.200000000000000000e+01

FAILED TRIALS — size and last line inspection

h0/s8/p1/r2/open_hand
  Size:      262,144 bytes (78.5% of reference)
  Lines:     1276
  Last line: -2.000000000000000000e+00,-6.000000000000000000e+00,0.000000000000000000e+00,6.000000000000000000e+

h0/s8/p5/r2/eversion
  Size:      262,144 bytes (78.5% of reference)
  Lines:     1276
  Last line: -5.000000000000000000e+00,-1.100000000000000000e+01,3.600000000000000000e

h1/s0/p1/r2/pinch_middlefinger
  Size:      262,144 bytes (78

## Verdict and recommendation

If the failed files are clearly truncated (size much smaller than reference, last line looks incomplete), the cleanest action is to **exclude them from the metadata** and proceed with the remaining 24,457 trials (99.88%). Trying to "recover" a partial signal would introduce artifacts.

We'll tag them in metadata and drop them downstream.

In [3]:
# Distribution of failed files across subjects — is corruption localized?
print("Failed trials grouped by subject:")
print(failed_df["subject"].value_counts())
print()
print("Failed trials grouped by (subject, session):")
print(failed_df.groupby(["subject", "session"]).size())

Failed trials grouped by subject:
subject
h4     8
h1     4
h5     3
h0     2
h2     2
h10    1
h13    1
h20    1
h23    1
h24    1
h30    1
h33    1
h7     1
h8     1
h9     1
Name: count, dtype: int64

Failed trials grouped by (subject, session):
subject  session
h0       8          2
h1       0          2
         1          1
         4          1
h10      2          1
h13      1          1
h2       4          1
         5          1
h20      0          1
h23      0          1
h24      0          1
h30      0          1
h33      0          1
h4       0          1
         1          1
         5          1
         6          1
         8          3
         9          1
h5       4          1
         5          1
         8          1
h7       1          1
h8       1          1
h9       0          1
dtype: int64
